In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load datasets
nchs = pd.read_csv('../../data/NCHS_Mortality_Clean.csv')
dea = pd.read_csv('../../data/dea_full_interpolated.csv')
ukcpr = pd.read_csv('../../data/UKCPR_cleaned.csv')

In [3]:
# Add state abbrev to NCHS
state_map = {
    'ALABAMA': 'AL', 'ALASKA': 'AK', 'ARIZONA': 'AZ', 'ARKANSAS': 'AR', 'CALIFORNIA': 'CA',
    'COLORADO': 'CO', 'CONNECTICUT': 'CT', 'DELAWARE': 'DE', 'FLORIDA': 'FL', 'GEORGIA': 'GA',
    'HAWAII': 'HI', 'IDAHO': 'ID', 'ILLINOIS': 'IL', 'INDIANA': 'IN', 'IOWA': 'IA',
    'KANSAS': 'KS', 'KENTUCKY': 'KY', 'LOUISIANA': 'LA', 'MAINE': 'ME', 'MARYLAND': 'MD',
    'MASSACHUSETTS': 'MA', 'MICHIGAN': 'MI', 'MINNESOTA': 'MN', 'MISSISSIPPI': 'MS', 'MISSOURI': 'MO',
    'MONTANA': 'MT', 'NEBRASKA': 'NE', 'NEVADA': 'NV', 'NEW HAMPSHIRE': 'NH', 'NEW JERSEY': 'NJ',
    'NEW MEXICO': 'NM', 'NEW YORK': 'NY', 'NORTH CAROLINA': 'NC', 'NORTH DAKOTA': 'ND', 'OHIO': 'OH',
    'OKLAHOMA': 'OK', 'OREGON': 'OR', 'PENNSYLVANIA': 'PA', 'RHODE ISLAND': 'RI', 'SOUTH CAROLINA': 'SC',
    'SOUTH DAKOTA': 'SD', 'TENNESSEE': 'TN', 'TEXAS': 'TX', 'UTAH': 'UT', 'VERMONT': 'VT',
    'VIRGINIA': 'VA', 'WASHINGTON': 'WA', 'WEST VIRGINIA': 'WV', 'WISCONSIN': 'WI', 'WYOMING': 'WY',
    'DISTRICT OF COLUMBIA': 'DC'
}


# Remove national data and DC from NCHS since DEA data is state-level
nchs = nchs[(nchs['state'] != 'United States')].reset_index(drop=True)
# nchs = nchs[(nchs['state'] != 'DC')].reset_index(drop=True)# Strip spaces and force uppercase before filtering
nchs = nchs[nchs['state'].str.strip().str.upper() != 'DC'].reset_index(drop=True)

# Verify it worked
print(f"Remaining states: {nchs['state'].unique()}")
# Map states
nchs['state'] = nchs['state'].str.upper().str.strip()
nchs['state'] = nchs['state'].map(state_map)

# Remove 1999 from NCHS since DEA data starts in 2000
nchs = nchs[nchs['year'] >= 2000].reset_index(drop=True)

nchs.head()

Remaining states: ['Alabama' 'Alaska' 'Arizona' 'Arkansas' 'California' 'Colorado'
 'Connecticut' 'Delaware' 'District of Columbia' 'Florida' 'Georgia'
 'Hawaii' 'Idaho' 'Illinois' 'Indiana' 'Iowa' 'Kansas' 'Kentucky'
 'Louisiana' 'Maine' 'Maryland' 'Massachusetts' 'Michigan' 'Minnesota'
 'Mississippi' 'Missouri' 'Montana' 'Nebraska' 'Nevada' 'New Hampshire'
 'New Jersey' 'New Mexico' 'New York' 'North Carolina' 'North Dakota'
 'Ohio' 'Oklahoma' 'Oregon' 'Pennsylvania' 'Rhode Island' 'South Carolina'
 'South Dakota' 'Tennessee' 'Texas' 'Utah' 'Vermont' 'Virginia'
 'Washington' 'West Virginia' 'Wisconsin' 'Wyoming']


,state,year,sex,age_group,age_group_detail,race,death_rate,death_rate_log,us_crude_rate,us_age_adjusted_rate,deaths,population
0,AL,2000,Both Sexes,All Ages,All Ages,All Races,4.4857,0.739232,6.1882,6.1749,197,4447100
1,AL,2001,Both Sexes,All Ages,All Ages,All Races,4.8915,0.770226,6.8057,6.7922,216,4467634
2,AL,2002,Both Sexes,All Ages,All Ages,All Races,4.7619,0.760566,8.1766,8.1957,211,4480089
3,AL,2003,Both Sexes,All Ages,All Ages,All Races,4.4333,0.735064,8.8881,8.8765,197,4503491
4,AL,2004,Both Sexes,All Ages,All Ages,All Races,6.3542,0.866535,9.3660,9.3831,283,4530729


In [4]:
nchs.sex.value_counts()

sex
Both Sexes    867
Name: count, dtype: int64

In [5]:
# 1. Count DC rows before
print(f"DC rows before: {len(nchs[nchs['state'] == 'DC'])}")

# 2. Filter using the exact opposite of what worked for your .loc
# We add .copy() to ensure it's a fresh dataframe
nchs = nchs.loc[nchs['state'] != "DC"].copy().reset_index(drop=True)

# 3. Count DC rows after
print(f"DC rows after: {len(nchs[nchs['state'] == 'DC'])}")

DC rows before: 17
DC rows after: 0


In [6]:
dea.head()

,state,year,hydro_gms,oxy_gms
0,AK,2000,27018.400,74395.720
1,AK,2001,30705.658,78212.956
2,AK,2002,34392.916,82030.192
3,AK,2003,38080.174,85847.428
4,AK,2004,41767.432,89664.664


In [7]:
# Drop 1999 from UKCPR
ukcpr = ukcpr.loc[ukcpr.year >= 2000].reset_index(drop=True)

ukcpr.head()

,state_name,state_abbr,year,poverty_rate,unempl_rate,gsp,min_wage,snap_rate,medicaid_rate,aca_exp,gov_dem
0,Alabama,AL,2000,13.3,4.6,120132.9,5.15,0.088958,0.126888,0.0,1
1,Alaska,AK,2000,7.6,6.4,26815.8,5.65,0.059755,0.132430,0.0,1
2,Arizona,AZ,2000,11.7,4.0,164609.9,5.15,0.050189,0.093786,0.0,0
3,Arkansas,AR,2000,16.5,4.3,68740.4,5.15,0.092053,0.146352,0.0,0
4,California,CA,2000,12.7,4.9,1366166.5,6.25,0.053893,0.181500,0.0,1


In [8]:
# Join NCHS and DEA on state and year
nchs_dea = pd.merge(
    nchs, 
    dea, 
    on=['year', 'state'], 
    how='left'
)

missing_count = nchs_dea['oxy_gms'].isna().sum()
if missing_count > 0:
    print(f"Warning: {missing_count} rows in NCHS did not find matching DEA data.")
else:
    print("Perfect match! No missing supply data in the final merge.")

Perfect match! No missing supply data in the final merge.


In [ ]:
# Convert hydro_gms, oxy_gms to be rate per 100k population
nchs_dea['hydro_gms'] = (nchs_dea['hydro_gms'] / nchs_dea['population']) * 100000 
nchs_dea['oxy_gms'] = (nchs_dea['oxy_gms'] / nchs_dea['population']) * 100000
nchs_dea['fent_gms'] = (nchs_dea['fent_gms'] / nchs_dea['population']) * 100000

In [10]:
# Merge with UKCPR on state and year
df_final = pd.merge(
    nchs_dea,
    ukcpr,
    left_on=['year', 'state'],
    right_on=['year', 'state_abbr'],
    how = 'left'
)

cols_to_keep = ['year', 'state', 'sex', 'age_group', 'age_group_detail', 'race', 'hydro_gms',
                'oxy_gms', 'unempl_rate','poverty_rate', 'gsp', 'min_wage',
                'snap_rate', 'medicaid_rate', 'aca_exp', 'gov_dem', 'death_rate']

df_final = df_final[cols_to_keep]
df_final.head()


,year,state,sex,age_group,age_group_detail,race,hydro_gms,oxy_gms,unempl_rate,poverty_rate,gsp,min_wage,snap_rate,medicaid_rate,aca_exp,gov_dem,death_rate
0,2000,AL,Both Sexes,All Ages,All Ages,All Races,10664.777495,6724.566122,4.6,13.3,120132.9,5.15,0.088958,0.126888,0.0,1,4.4857
1,2001,AL,Both Sexes,All Ages,All Ages,All Races,12065.615178,7317.474753,5.1,15.9,123035.3,5.15,0.092060,0.150865,0.0,1,4.8915
2,2002,AL,Both Sexes,All Ages,All Ages,All Races,13477.895908,7919.213212,5.9,14.5,128117.4,5.15,0.099004,0.164526,0.0,1,4.7619
3,2003,AL,Both Sexes,All Ages,All Ages,All Races,14846.170182,8496.910730,6.0,15.0,133969.3,5.15,0.104822,0.166062,0.0,0,4.4333
4,2004,AL,Both Sexes,All Ages,All Ages,All Races,16186.581541,9060.957298,5.7,16.9,146886.7,5.15,0.109826,0.172655,0.0,0,6.3542


In [11]:
df_final.head()

,year,state,sex,age_group,age_group_detail,race,hydro_gms,oxy_gms,unempl_rate,poverty_rate,gsp,min_wage,snap_rate,medicaid_rate,aca_exp,gov_dem,death_rate
0,2000,AL,Both Sexes,All Ages,All Ages,All Races,10664.777495,6724.566122,4.6,13.3,120132.9,5.15,0.088958,0.126888,0.0,1,4.4857
1,2001,AL,Both Sexes,All Ages,All Ages,All Races,12065.615178,7317.474753,5.1,15.9,123035.3,5.15,0.092060,0.150865,0.0,1,4.8915
2,2002,AL,Both Sexes,All Ages,All Ages,All Races,13477.895908,7919.213212,5.9,14.5,128117.4,5.15,0.099004,0.164526,0.0,1,4.7619
3,2003,AL,Both Sexes,All Ages,All Ages,All Races,14846.170182,8496.910730,6.0,15.0,133969.3,5.15,0.104822,0.166062,0.0,0,4.4333
4,2004,AL,Both Sexes,All Ages,All Ages,All Races,16186.581541,9060.957298,5.7,16.9,146886.7,5.15,0.109826,0.172655,0.0,0,6.3542


In [12]:
# Export final dataset
df_final.to_csv('../../data/death_rate.csv', index=False)